# <div align="center"> SPECIAL TOPICS III </div>
# <div align="center"> Machine Learning And Econometrics  </div>
## <div align="center"> ECO 4971/6971  </div>
#### <div align="center">Class 10 - Model Selection</div>
<div align="center"> Jonathan Holmes, (he/him)</div>

# Subset Selection
$$Y = \beta_0+\beta_1X_1+\beta_2X_2+ ... + \beta_pX_p + \epsilon$$
- The issue it that adding predictors will always weakly improve the in sample prediction
    - But at the expense of out of sample prediction
- It is therefore important to limit the number of predictors to those actually related to Y
- __Subset Selection__ _is the process of identifying the p predictors actually related to Y._

## Best Subset Selection
- Here is the algorithm to select the best subset given a dataset with p predictors
- We will apply this algorithm to the Credit dataset

Algorithm:

1. Let $\mathscr{M}_0$ denote the null model , which contains no predictors. This
model simply predicts the sample mean for each observation.
2. For k = 1, 2, . . .p:
    - a. Fit all ${P\choose k}$ containing exactly $k$ predictors
    - b. Pick the best among these ${P\choose k}$ and call it $\mathscr{M}_k$. Here best is defined as having the smallest RSS, or equivalently largest R2.
3. Select a single best model from among $\mathscr{M}_0$, . . . ,$\mathscr{M}_p$ using crossvalidated prediction error (MSE), AIC, BIC, or adjusted $R^2$

### The null model
- Let's start with the null model
- This a model using no predictors and a single parameter: the intercept

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

import sklearn.linear_model as skl_lm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

from itertools import permutations
from pathlib import Path

CACHE_PATH = Path("best_subset_selection_credit_cache.pkl")
CACHE_URL = "https://www.dropbox.com/scl/fi/ndvbvik7oedea5mf8uq5u/best_subset_selection_credit_cache.pkl?rlkey=bzx6d7q4bb04vi58l4poqz8q7&dl=1"


In [ ]:
df=pd.read_csv("https://www.dropbox.com/scl/fi/dlvn95r99i6da5oc7ha82/Credit.csv?rlkey=rbnd58yh0s06lu8hmyj3esjzi&dl=1")
df['Own']=df['Own'].replace({"No":0,  "Yes":1})
df['Student']=df['Student'].replace({"No":0,  "Yes":1})
df['Married']=df['Married'].replace({"No":0,  "Yes":1})
region_dummies = pd.get_dummies(df['Region'], drop_first=True).astype(int)
df=df.join(region_dummies)
df.pop("Region")
display(df.shape)
df.head()

$\mathscr{M}_0$

In [ ]:
# M0
results = smf.ols('Balance ~ 1 ', data=df).fit()
# Inspect the results
print(results.summary())

In [ ]:
# get y and X
y=df['Balance'] # y
X=df.loc[:,~df.columns.str.contains('Balance')] # X
X.head()

$\mathscr{M}_1$

In [ ]:
# initiate empty list to store R2
R2= 0
best_predictor="none"
X=df.loc[:,~df.columns.str.contains('Balance')]
y=df.loc[:,df.columns.str.contains('Balance')]
X['intercept']=1


# for each predictor in X
for k in range(X.shape[1]-1): # minus 1 because last column is intercept
    results = sm.OLS(y, X.iloc[:,[k,-1]]).fit() # fit predictor in position p and intercept (in last position, -1)
    print(f"R\u00b2 for predictor {X.columns[k]} is {round(results.rsquared,6):f}")
    if results.rsquared>R2:
        R2=results.rsquared ; best_predictor=X.columns[k]
print("\nBest predictor for M1 is: {} with {}.".format(best_predictor,round(R2,3)))

# In-Class Exercise

Q1: Which single variable does the best job predicting `Balance`?

Q2: How many possible subsets are there if you have:
- 1 predictor
- 2 predictors
- 3 predictors
- 4 predictors
- 5 predictors


In [ ]:
"""
DFs=[]
#for p in range(X.shape[1]): # uncomment here if you want to do it for all RUN AT YOUR OWN RISK
for p in range(9):
    R2=0
    best_predictor="none"
    R2_list=[]; adj_R2_list=[] ; aic=[] ; bic=[] ; models=[]
    bestmodel=[-1]
    for k in permutations(list(range(X.shape[1]-1)), p):
        model=list(k)
        model.append(-1)
        results = sm.OLS(y, X.iloc[:,model]).fit() # fit predictors from model
        # append statistics to list
        R2_list.append(results.rsquared) ; adj_R2_list.append(results.rsquared_adj) 
        aic.append(results.aic); bic.append(results.bic) ; models.append(','.join(X.columns[model]))

        if results.rsquared_adj>R2:
            bestmodel=model
            #model.pop(-1)
            R2=results.rsquared_adj 
    DFs.append(pd.DataFrame(data={'R2':R2_list,'Adjusted R2':adj_R2_list, "AIC":aic, "BIC":bic,'Predictors':p,'model':models}))
    
    print("Best predictor for M{} is: {} with {}.".format(p,', '.join(X.columns[bestmodel]),round(R2,3)))
    
dd=pd.concat(DFs, ignore_index=True)
dd.to_pickle(CACHE_PATH)
dd.head()
"""
p = 8

In [ ]:
if CACHE_PATH.is_file():
    dd = pd.read_pickle(CACHE_PATH)
    print(f"Loaded {len(dd):,} rows from local cache: {CACHE_PATH.resolve()}")
else:
    print("Local cache not found. Loading precomputed cache from Dropbox...")
    dd = pd.read_pickle(CACHE_URL)
    print(f"Loaded {len(dd):,} rows from Dropbox cache URL.")

dd.head()


## Computational challenge

- Best subset selection is simple and intuitive.
- But the number of candidate models grows very quickly as the number of predictors increases.
- In general, with $p$ predictors, there are $2^p$ possible subset models.
  - If $p=10$, that is about 1,000 models.
  - If $p=20$, that is more than 1,000,000 models.


![title](credit_10predictors.png)

In [ ]:
dd_best=dd.groupby(['Predictors']).agg({"R2":np.max, "Adjusted R2":np.max, "AIC":np.min, "BIC":np.min}).reset_index()
dd_best

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(10,10),sharex=True)
axes = axes.ravel() # access axes with a single position instead of 2
for i, statistics in enumerate(['R2',"Adjusted R2","AIC","BIC"]):
    sns.scatterplot(x='Predictors',y=statistics,data=dd,ax=axes[i], color='gray',marker='.',alpha=.3)
    sns.lineplot(x='Predictors',y=statistics,data=dd_best,ax=axes[i], color='darkorange')
    sns.scatterplot(x='Predictors',y=statistics,data=dd_best,ax=axes[i], color='darkgreen')
    axes[i].set_ylabel(statistics)
    axes[i].set_xticks(np.arange(p+1))
    
fig.suptitle("In-Sample Statistics")
fig.tight_layout()
plt.show()

### Step 3 of Best Subset Selection
- You can think of this stage in terms of last lecture:
    - You have many models (last time many polynomial models), one for each p
    - But they all maximize the in sample fit
- In step 3, you can do this based on the adjusted-$R^2$, AIC, BIC or you can use CV techniques to find the model that gives the __best out of sample prediction__

In [ ]:
dd['best']=dd.groupby(['Predictors'])['Adjusted R2'].transform(np.max)
dd_best2=dd.loc[dd['Adjusted R2']==dd['best']].groupby('Predictors').first().reset_index()
dd_best2.head(10)

In [ ]:
models=["intercept",["Rating","intercept"], 
        ["Income","Rating","intercept"],
        ["Income","Rating","Student","intercept"],
        ["Income","Limit","Cards","Student","intercept"],
        ["Income","Limit","Rating","Cards","Student","intercept"],
        ["Income","Limit","Rating","Cards","Age","Student","intercept"],
        ["Income","Limit","Rating","Cards","Age","Own","Student","intercept"]]

In [ ]:
df['intercept']=1

In [ ]:
# Best number of predictors using Cross Validation
# use best model from precedent exercise to speed up the code
kfold=5
DFs=[]
kf = KFold(n_splits=kfold, random_state=1706, shuffle=True)

for i,m in enumerate(models):
    MSEs=[] # empty list of MSE scores
    X=df[m].values
    y=df['Balance'].values
    for train_index, test_index in kf.split(X):
        if i==0:
            X_train, X_test = X[train_index].reshape(-1, 1), X[test_index].reshape(-1, 1)
        else:
            X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        # regression
        reg = LinearRegression() # initiate the regression class
        reg.fit(X_train,y_train) # fit the data
        # Out of Sample MSE:
        mse=mean_squared_error(y_test, reg.predict(X_test))
        MSEs.append(mse)
    DFs.append(pd.DataFrame({'Predictors':i,'MSE':MSEs}))
    
MSE_scores=pd.concat(DFs)
mse=MSE_scores.groupby('Predictors').mean().reset_index()    
mse

In [ ]:
dd_best=dd_best.merge(mse, on='Predictors')
dd_best.head()

In [ ]:
# Best number of predictors for other statistics
adj_R2_best=int(dd_best2.loc[dd_best2['Adjusted R2']==dd_best2['Adjusted R2'].max(),'Predictors'])
aic_best=int(dd_best2.loc[dd_best2['AIC']==dd_best2['AIC'].min(),'Predictors'])
bic_best=int(dd_best2.loc[dd_best2['BIC']==dd_best2['BIC'].min(),'Predictors'])
mse_best=int(mse.loc[mse.MSE==mse.MSE.min(),'Predictors'])
best_preds=[adj_R2_best,aic_best,bic_best,mse_best]
print(f"Best number of parameters for\nAdjusted R-square: {adj_R2_best}\nAIC: {aic_best}\nBIC: {bic_best}\n10-fold CV:{mse_best} ")

fig, axes = plt.subplots(2,2, figsize=(12,12),sharex=True)
axes = axes.ravel() # access axes with a single position instead of 2
for i, statistics in enumerate(["Adjusted R2","AIC","BIC", "MSE"]):
    sns.lineplot(x='Predictors',y=statistics,data=dd_best,ax=axes[i], color='darkorange')
    sns.scatterplot(x='Predictors',y=statistics,data=dd_best,ax=axes[i], color='darkgreen')
    # axes[i].axvline(best_preds[i], color='k')
    axes[i].scatter(x=best_preds[i],y=float(dd_best.loc[best_preds[i],statistics]),marker='X',color='red',s=100)
    axes[i].set_ylabel(statistics)
    axes[i].set_xticks(np.arange(p+1))
fig.suptitle("Best number of parameters by technique")
plt.show()

## Solution to the computational challenge: Forward Stepwise Selection

1. Let $\mathscr{M}_0$ be the null model (intercept only).
2. For $k = 0, \dots, p-1$:
   - Consider all $p-k$ models formed by adding one predictor to $\mathscr{M}_k$.
   - Choose the best model and call it $\mathscr{M}_{k+1}$.
   - “Best” can mean smallest RSS or largest $R^2$.
3. Choose a final model from $\mathscr{M}_0, \dots, \mathscr{M}_p$ using cross-validated prediction error, AIC, BIC, or adjusted $R^2$.


## Solution to the computational challenge: Backward Stepwise Selection

1. Let $\mathscr{M}_p$ be the full model (all predictors).
2. For $k = p, p-1, \dots, 1$:
   - Consider all $k$ models formed by removing one predictor from $\mathscr{M}_k$.
   - Choose the best model and call it $\mathscr{M}_{k-1}$.
   - “Best” can mean smallest RSS or largest $R^2$.
3. Choose a final model from $\mathscr{M}_0, \dots, \mathscr{M}_p$ using cross-validated prediction error, AIC, BIC, or adjusted $R^2$.


# When to use forward vs. backward stepwise selection?

**Forward stepwise selection:**
- Useful when the dataset has many predictors and exhaustive search is computationally expensive.

**Backward stepwise selection:**
- Often performs better when predictors are highly collinear (if fitting the full model is feasible).


## Taking stock
- Overfitting risk increases as model complexity grows.
- Subset methods help control complexity by penalizing large models.
- The goal is strong out-of-sample prediction with a parsimonious set of predictors.


# Shrinkage methods
- In the `Credit` dataset, many predictors look useful for `Balance`.
- Instead of searching all subsets, we can keep all predictors and **shrink** less useful coefficients.
- This is called **regularization**.
- Intuition: important predictors keep larger coefficients; redundant predictors are pushed toward zero.


# Ridge regression
- For a model with $p$ predictors, OLS chooses $eta_0, eta_1, \ldots, eta_p$ to minimize:

$$\Large 	ext{RSS} = \sum_{i=1}^n \Big(y_i - eta_0 - \sum_{j=1}^p eta_jx_{ij}\Big)^2$$


- Ridge is similar to OLS, but adds a penalty on coefficient size.
- Ridge estimates $\hat{\beta}^{R}$ minimize:

$$
\sum_{i=1}^n \Big(y_i - \beta_0 - \sum_{j=1}^p \beta_jx_{ij}\Big)^2 + \lambda \sum_{j=1}^p\beta_j^2
= \text{RSS} + \lambda \sum_{j=1}^p\beta_j^2
$$


## Ridge regression: shrinkage intuition
- $\lambda$ is a **tuning parameter** chosen outside the optimization.
- Ridge still tries to fit the data well (small RSS).
- The penalty term $\lambda\sum_{j=1}^peta_j^2$ is small when coefficients are small ($eta_0$ is not penalized).
- If $\lambda=0$, Ridge is OLS.
- As $\lambda	o\infty$, coefficients shrink toward zero.
- We usually choose $\lambda$ with cross-validation.


## Standardizing the dataset

- Standardization is a common preprocessing step in machine learning (see [scikit-learn preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)).
- It centers each feature (subtract mean) and scales by standard deviation.
- Ridge and Lasso are sensitive to feature scale because the penalty is applied to coefficients.
- If one feature has much larger variance, it can dominate the objective.
- In OLS, scaling a predictor by $c$ rescales its coefficient by $1/c$ but does not change fitted values.
- In Ridge/Lasso, coefficient shrinkage depends on both $\lambda$ and predictor scaling, so standardization is especially important.


# In-Class Exercise

Q3: Why do we call "Ridge" regression a "Shrinkage" estimator? 

Q4: Suppose that we are predicting a farm harvest using the following regression

$$ \text{TotalHarvest} = \beta_0 + \beta_1 \text{AverageTemperature} + \beta_2 \text{AverageRainfall} + u $$
    
a) If I run OLS two times with temperature measured in degrees celcius and degrees kelvin, what will change? 

b) If I ran Ridge regression two times measured in degrees celcius and degrees kelvin, what will change? 

    



In [ ]:
df.head()

In [ ]:
# import the preprocessing module from sklearn
from sklearn import preprocessing
X_train =df[['Income','Limit','Rating','Student','Cards','Age','Education','Married','Own', 'West', 'South']]
scaler = preprocessing.StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_train.mean(axis=0) , X_train.std(axis=0)

### What happens as $\lambda$ changes?
We now trace how coefficient estimates move as the penalty strength increases.


In [ ]:
from sklearn.linear_model import Ridge

lambdas = 10**np.linspace(10,-2,100)*0.5
ridge = Ridge()
coefs = []

for 𝜆 in lambdas:
    ridge.set_params(alpha=𝜆)
    ridge.fit(X_train, y)
    coefs.append(ridge.coef_)
ridge_results=pd.DataFrame(coefs,columns=['Income','Limit','Rating','Student','Cards','Age','Education','Married','Own', 'West', 'South'])
ridge_results['Lambda']=lambdas  
ridge_results=pd.melt(ridge_results,id_vars=['Lambda'], var_name='Beta', value_name='Estimate')
ridge_results.head()                        


In [ ]:
fig, ax =plt.subplots(1,1, figsize=(7,7))
sns.lineplot(x='Lambda', y='Estimate', hue='Beta',data=ridge_results)
ax.set_xscale('log')
ax.axhline(0,color='k',linestyle=":")
plt.axis('tight')
plt.xlabel(r'$\lambda$', fontsize=20)
plt.ylabel('Standardized Coefficients',fontsize=20)
plt.title('Ridge coefficients as a function of the regularization');

plt.title('Plot B', fontsize=20)

## Ridge and model selection
- Ridge is much faster than exhaustive best-subset search.
- But Ridge does **not** select a subset: it keeps all predictors and shrinks them.
- This is often great for prediction, but can reduce interpretability when $p$ is large.
- In the `Credit` data, variables like `Income`, `Limit`, `Rating`, and `Student` appear most influential.
- If interpretability is key, we may prefer a method that can set some coefficients exactly to zero.


## Lasso regression
- Lasso is an alternative to Ridge that can shrink coefficients **exactly to zero**.
- Lasso estimates $\hat{eta}^{L}_{\lambda}$ minimize:

egin{gather}
\Large	ext{RSS} + \lambda \sum_{j=1}^p\left|eta_jight|
\end{gather}


## Lasso and model selection
- Like Ridge, Lasso shrinks coefficient estimates.
- With the $\ell_1$ penalty, some coefficients become exactly zero when $\lambda$ is large enough.
- So Lasso performs **automatic variable selection**.
- This yields **sparse models** (few non-zero coefficients).


In [ ]:
from sklearn.linear_model import Lasso

lambdas = 10**np.linspace(10,-1.5,100)*0.5
lasso = Lasso()
coefs = []


for 𝜆 in lambdas:
    lasso.set_params(alpha=𝜆)
    lasso.fit(X_train, y)
    coefs.append(lasso.coef_)
lasso_results=pd.DataFrame(coefs,columns=['Income','Limit','Rating','Student','Cards','Age','Education','Gender','Married','Asian','Caucasian'])
lasso_results['Lambda']=lambdas  
lasso_results=pd.melt(lasso_results,id_vars=['Lambda'], var_name='Beta', value_name='Estimate')
lasso_results.head()                        


In [ ]:
fig, ax =plt.subplots(1,1, figsize=(7,7))
ax = plt.gca()
sns.lineplot(x='Lambda', y='Estimate', hue='Beta',data=lasso_results)
ax.set_xscale('log')
ax.axhline(0,color='k',linestyle=":")
plt.axis('tight')
plt.xlabel(r'$\lambda$', fontsize=20)
plt.ylabel('Standardized Coefficients',fontsize=20)
#plt.title('Lasso coefficients as a function of the regularization');

plt.title('Plot A', fontsize=20)

## Another formulation of Ridge and Lasso

- Ridge and Lasso can also be written as constrained optimization problems.
- This is equivalent to the penalty form for an appropriate mapping between $\lambda$ and $s$.

__Ridge__:
$$
\min_{\mathbf{\beta}} \left\{ \sum_{i=1}^n \Big(y_i - \beta_0 - \sum_{j=1}^p \beta_jx_{ij}\Big)^2 \right\}
\quad \text{subject to } \sum_{j=1}^p\beta_j^2 \leq s
$$

__Lasso__:
$$
\min_{\mathbf{\beta}} \left\{ \sum_{i=1}^n \Big(y_i - \beta_0 - \sum_{j=1}^p \beta_jx_{ij}\Big)^2 \right\}
\quad \text{subject to } \sum_{j=1}^p\left|\beta_j\right| \leq s
$$


- For each $\lambda$, there is an equivalent constraint size $s$.
- When $p=2$:
  - Lasso: minimize RSS subject to $|\beta_1|+|\beta_2|\le s$
  - Ridge: minimize RSS subject to $\beta_1^2+\beta_2^2\le s$
- Think of $s$ as a coefficient “budget.”
- Large $s$: weak constraint (close to OLS).
- Small $s$: strong constraint (more shrinkage).


### Geometry behind Ridge vs Lasso

For **two predictors** ($p=2$), you can picture **Ridge** and **Lasso** as:

- **Minimize RSS** (same “bowl-shaped” surface in $(\beta_1,\beta_2)$ space — shown as **elliptical contours**),
- **subject to a budget constraint** on the coefficients:
  - **Lasso**: $|\beta_1| + |\beta_2| \le s$ → a **diamond** (rotated square) in 2D.
  - **Ridge**: $\beta_1^2 + \beta_2^2 \le s$ → a **disk** (circle) in 2D.

The **constrained solution** is typically the **first contour that touches the feasible set**. Corners of the diamond are why Lasso can **set a coefficient exactly to zero** (variable selection), while the smooth circle in Ridge usually gives **all coefficients slightly shrunk** but not exactly zero.

The **next cell generates this picture with code** (so you can change $s$, $\hat{\beta}$, etc.).

In [ ]:
# Ridge vs. Lasso: geometric intuition and constrained optimization

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize


## OLS vs. Ridge and Lasso
- In the geometry plot, the OLS solution $\hat{\beta}$ is often outside the feasible region.
- Ellipses centered at $\hat{\beta}$ are constant-RSS contours.
- Ridge/Lasso solutions are the first contour touching their constraint set.
- Ridge’s smooth circle usually gives non-zero coefficients.
- Lasso’s corners often lead to axis intersections, so one coefficient can be exactly zero.
- In the illustrated case, the Lasso solution lies on $\beta_1=0$, so only $\beta_2$ remains.


## Selecting the tuning parameter

- Model performance depends on the value of $\lambda$.
- Our goal is strong out-of-sample prediction.
- So we choose $\lambda$ using cross-validation.


In [ ]:
# Credit data: Ridge tuning curve via k-fold cross-validation
# Uses the standardized design matrix X_train from earlier cells.

from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

y_cv = df["Balance"].to_numpy()
X_cv = X_train  # already standardized features

n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=1706)

# Grid for lambda (Ridge's alpha); log-spaced is easier to read on a plot
lambdas_cv = np.logspace(-2, 5.5, 80)

cv_mse = []
for lam in lambdas_cv:
    fold_mse = []
    for train_idx, val_idx in kf.split(X_cv):
        model = Ridge(alpha=lam)
        model.fit(X_cv[train_idx], y_cv[train_idx])
        pred = model.predict(X_cv[val_idx])
        fold_mse.append(mean_squared_error(y_cv[val_idx], pred))
    cv_mse.append(np.mean(fold_mse))

cv_mse = np.array(cv_mse)
best_lam = lambdas_cv[np.argmin(cv_mse)]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lambdas_cv, cv_mse, color="darkorange", linewidth=2)
ax.scatter([best_lam], [np.min(cv_mse)], color="k", zorder=3, label=r"min CV MSE ($\lambda$ = {:.4g})".format(best_lam))
ax.set_xscale("log")
ax.set_xlabel(r"Tuning parameter $\lambda$ (Ridge `alpha`)")
ax.set_ylabel(f"{n_splits}-fold cross-validated MSE")
ax.set_title("Credit data: choosing $\\lambda$ for Ridge regression")
ax.legend(loc="best", frameon=True)
ax.grid(True, linestyle=":", alpha=0.45)
plt.tight_layout()
plt.show()

print(f"Lambda with minimum CV MSE: {best_lam:.6g}")
print(f"Minimum CV MSE: {np.min(cv_mse):.2f}")

### Choosing Ridge $\lambda$ by cross-validation

- Split the `Credit` data into train/validation folds.
- For each candidate $\lambda$, fit Ridge on training folds and compute validation MSE.
- Average validation MSE across folds.
- Select the $\lambda$ with the lowest cross-validated MSE (or a nearby larger value for extra shrinkage/simplicity).


![image.png](lasso_lambda.png)

# In-Class Exercise

Q5: What is the difference between Lasso and Ridge regression?

Q6: How would you decide among Lasso, Ridge, and OLS in practice?


In [ ]:
# Here is an example of using built-in Python commands to do cross-validated Lasso and Ridge.
#
# RidgeCV only matches the *manual* Ridge CV curve above when these line up:
# same λ grid as `lambdas_cv`, same KFold (shuffle + random_state), same 1d y, and MSE for picking α.

from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

Xdata = X_train
# Xdata = X

y_reg = df["Balance"].to_numpy()

# Must match the "Ridge tuning curve" cell exactly (change both together if you edit the grid or KFold).
lambdas_cv = np.logspace(-2, 5.5, 80)
ridge_cv = KFold(n_splits=10, shuffle=True, random_state=1706)

results_lasso = LassoCV(cv=10, random_state=31415).fit(Xdata, y)

print(
    "Lasso CV selected lambda = {} with an $R^2$ of {}".format(
        round(results_lasso.alpha_, 2),
        round(results_lasso.score(Xdata, y), 2),
    )
)

results_ridge = RidgeCV(
    alphas=lambdas_cv,
    cv=ridge_cv,
    scoring="neg_mean_squared_error",
).fit(Xdata, y_reg)

print(
    "RidgeCV selected λ = {} | in-sample $R^2$ = {}".format(
        results_ridge.alpha_,
        round(results_ridge.score(Xdata, y_reg), 2),
    )
)

try:
    print(f"Manual argmin λ from curve cell: {best_lam:.6g}")
    print(f"Same as RidgeCV? {np.isclose(results_ridge.alpha_, best_lam)}")
except NameError:
    print("Run the Ridge tuning-curve cell first so `best_lam` exists.")

